# 06 - Clustering Model Development

## Objective

This notebook develops and evaluates the customer segmentation clustering model
using the final preprocessed customer feature dataset.

The modelling stage will:

- load the registered clustering-input dataset;
- separate customer identifiers from modelling features;
- evaluate a range of candidate cluster counts;
- assess cluster quality using multiple internal validation metrics;
- select an appropriate number of customer segments;
- train the final clustering model;
- assign each customer to a cluster;
- evaluate cluster sizes and behavioural separation;
- persist the trained model and customer cluster assignments.

Because clustering is unsupervised, there is no target variable or traditional
accuracy measure. Model selection will therefore consider both statistical
cluster quality and business interpretability.

## 2. Clustering Evaluation Framework

Multiple clustering algorithms will be evaluated rather than assuming a single
method is appropriate for the customer segmentation problem.

Where applicable, candidate solutions will be compared using:

- **Silhouette Score:** measures how well customers fit within their assigned
  cluster compared with neighbouring clusters. Higher values are better.
- **Calinski-Harabasz Score:** compares between-cluster separation with
  within-cluster compactness. Higher values are better.
- **Davies-Bouldin Score:** measures similarity between clusters. Lower values
  are better.

Additional algorithm-specific measures will also be considered, including
K-Means inertia and Gaussian Mixture AIC/BIC.

Model selection will consider statistical quality alongside cluster size,
stability and business interpretability.

In [1]:
# necessary libraries
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import Data
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import AccountKeyConfiguration
import mltable
from mltable import MLTableHeaders, MLTableFileEncoding
import pandas as pd
import numpy as np

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

# Load the registered Azure ML Data Asset so that analysis is based on the
# governed MLTable rather than a local file path.
paid_purchase_asset = ml_client.data.get(
    name="online-retail-paid-purchases",
    version="1"
)

# Load the MLTable definition and materialise the dataset as a Pandas DataFrame
# for interactive profiling and exploratory analysis.
retail_table = mltable.load(paid_purchase_asset.path)
df_paid = retail_table.to_pandas_dataframe()

Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [2]:
# Retrieve the registered final clustering-input dataset from Azure ML.

clustering_asset = ml_client.data.get(
    name="online-retail-clustering-input",
    version="1"
)

print(clustering_asset.name)
print(clustering_asset.version)

online-retail-clustering-input
1


In [3]:
# Load the registered MLTable and materialise it as a Pandas DataFrame.

clustering_table = mltable.load(
    clustering_asset.path
)

df_clustering = clustering_table.to_pandas_dataframe()

print("Rows:", f"{df_clustering.shape[0]:,}")
print("Columns:", df_clustering.shape[1])

df_clustering.head()

Rows: 4,334
Columns: 9


,CustomerID,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
0,12346,1.460438,-0.951243,3.729311,-2.523812,-0.986461,6.873723,-0.270652,10.364965
1,12347,-2.038956,1.082163,1.429190,0.966888,1.777127,0.908605,-0.270652,-0.282114
2,12348,0.371964,0.392765,0.553997,-0.405407,1.148694,1.472557,4.005881,-0.282114
3,12349,-0.623847,-0.951243,0.565177,0.654210,-0.986461,1.558851,4.005881,-0.282114
4,12350,1.423017,-0.951243,-0.707943,-0.633185,-0.986461,0.261075,4.005881,-0.282114


In [4]:
# Preserve CustomerID separately for traceability.
# CustomerID must not be supplied to the clustering algorithm.

customer_ids = df_clustering["CustomerID"].copy()

X_model = df_clustering.drop(
    columns=["CustomerID"]
).copy()

print("Model matrix shape:", X_model.shape)
print("Number of customers:", len(customer_ids))

Model matrix shape: (4334, 8)
Number of customers: 4334


In [5]:
# Confirm that the modelling matrix is complete and contains only
# finite numeric values before clustering begins.

print("Missing values:", X_model.isna().sum().sum())
print("Infinite values:", np.isinf(X_model).sum().sum())

X_model.dtypes

Missing values: 0
Infinite values: 0


Recency                               float64
Frequency                             float64
MonetaryValue                         float64
UniqueProducts                        float64
CustomerTenureDays                    float64
AverageQuantityPerOrder               float64
PostageInvoiceShare                   float64
ObservedMerchandiseReturnValueRate    float64
dtype: object

In [6]:
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
    )

# Calculate common internal clustering metrics for a set of cluster labels.
# Using one function ensures that different algorithms are evaluated
# consistently throughout the modelling experiments.

def evaluate_clustering(X, labels):
    return {
        "SilhouetteScore": silhouette_score(X, labels),
        "CalinskiHarabaszScore": calinski_harabasz_score(X, labels),
        "DaviesBouldinScore": davies_bouldin_score(X, labels)
    }

candidate_clusters = range(2, 11)

print(list(candidate_clusters))

[2, 3, 4, 5, 6, 7, 8, 9, 10]


In [7]:
from sklearn.cluster import KMeans

# Store the evaluation results for each candidate number of clusters.
kmeans_results = []

# Train and evaluate a separate K-Means model for each candidate cluster count.
for k in candidate_clusters:

    # Create a reproducible K-Means model.
    # random_state ensures consistent results between runs.
    # n_init="auto" allows scikit-learn to determine the appropriate
    # number of centroid initialisations.
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    # Fit the model and assign each customer to a cluster.
    labels = kmeans.fit_predict(X_model)

    # Calculate the common clustering quality metrics.
    metrics = evaluate_clustering(
        X_model,
        labels
    )

    # Store both K-Means-specific and common evaluation metrics
    # so that different cluster counts can be compared consistently.
    kmeans_results.append({
        "Clusters": k,
        "Inertia": kmeans.inertia_,
        "SilhouetteScore": metrics["SilhouetteScore"],
        "CalinskiHarabaszScore": metrics["CalinskiHarabaszScore"],
        "DaviesBouldinScore": metrics["DaviesBouldinScore"]
    })

# Convert the collected results into a DataFrame for comparison.
kmeans_results = pd.DataFrame(kmeans_results)

# Display the evaluation results for candidate cluster counts.
kmeans_results.round(4)

,Clusters,Inertia,SilhouetteScore,CalinskiHarabaszScore,DaviesBouldinScore
0,2,23745.9333,0.3094,1993.2694,1.2678
1,3,21081.1937,0.2022,1396.1031,1.6446
2,4,17682.8025,0.2300,1386.7365,1.3540
3,5,15969.4232,0.2110,1267.4762,1.3586
4,6,13324.7816,0.2212,1386.7817,1.1955
5,7,12720.9498,0.2243,1244.4414,1.1798
6,8,11802.7801,0.2044,1197.4615,1.3166
7,9,11182.7067,0.2067,1135.5919,1.3110
8,10,10588.9698,0.1994,1092.7054,1.2645


In [8]:
from sklearn.mixture import GaussianMixture

# Store evaluation results for each candidate number of Gaussian components.
gmm_results = []

# Train and evaluate a Gaussian Mixture Model for each candidate
# number of customer segments.
for k in candidate_clusters:

    # Use full covariance matrices so each customer segment can have
    # its own shape and feature relationships.
    # Multiple initialisations reduce sensitivity to the starting solution.
    gmm = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=42,
        n_init=5,
        max_iter=500
    )

    # Fit the probabilistic clustering model.
    gmm.fit(X_model)

    # Assign each customer to the component with the highest
    # estimated membership probability.
    labels = gmm.predict(X_model)

    # Calculate the common clustering quality metrics so that GMM
    # can later be compared directly with K-Means.
    metrics = evaluate_clustering(
        X_model,
        labels
    )

    # Store both common clustering metrics and GMM-specific
    # information criteria.
    gmm_results.append({
        "Clusters": k,
        "AIC": gmm.aic(X_model),
        "BIC": gmm.bic(X_model),
        "SilhouetteScore": metrics["SilhouetteScore"],
        "CalinskiHarabaszScore": metrics["CalinskiHarabaszScore"],
        "DaviesBouldinScore": metrics["DaviesBouldinScore"],
        "Converged": gmm.converged_
    })

# Convert results into a DataFrame for comparison.
gmm_results = pd.DataFrame(gmm_results)

# Display the candidate GMM results.
gmm_results.round(4)

,Clusters,AIC,BIC,SilhouetteScore,CalinskiHarabaszScore,DaviesBouldinScore,Converged
0,2,22138.7817,22706.0896,0.3530,547.8576,1.4396,True
1,3,-17765.8511,-16911.7021,0.2269,946.5702,1.8515,True
2,4,-28255.0500,-27114.0599,0.0856,612.4266,2.5981,True
3,5,-46143.9059,-44716.0748,0.0925,544.1411,2.6144,True
4,6,-47651.5757,-45936.9034,0.0978,557.3131,2.3918,True
5,7,-49507.3448,-47505.8315,0.0835,485.0184,2.2968,True
6,8,-51457.8873,-49169.5329,0.0980,468.1029,2.5351,True
7,9,-54624.5966,-52049.4011,0.0760,442.6777,2.7888,True
8,10,-55489.7402,-52627.7037,0.0470,447.8005,2.8298,True


In [9]:
from sklearn.cluster import AgglomerativeClustering

# Store evaluation results for each candidate number of hierarchical clusters.
agglomerative_results = []

# Train and evaluate an Agglomerative Clustering model for each
# candidate number of customer segments.
for k in candidate_clusters:

    # Ward linkage merges clusters based on the increase in within-cluster
    # variance and is appropriate for the scaled Euclidean feature space.
    agglomerative = AgglomerativeClustering(
        n_clusters=k,
        linkage="ward"
    )

    # Fit the hierarchical clustering model and assign each customer
    # to a cluster.
    labels = agglomerative.fit_predict(X_model)

    # Calculate the same internal validation metrics used for
    # K-Means and Gaussian Mixture Models.
    metrics = evaluate_clustering(
        X_model,
        labels
    )

    # Store the results so candidate cluster counts can be compared.
    agglomerative_results.append({
        "Clusters": k,
        "SilhouetteScore": metrics["SilhouetteScore"],
        "CalinskiHarabaszScore": metrics["CalinskiHarabaszScore"],
        "DaviesBouldinScore": metrics["DaviesBouldinScore"]
    })

# Convert the collected results into a DataFrame for comparison.
agglomerative_results = pd.DataFrame(
    agglomerative_results
)

# Display the evaluation results for each candidate cluster count.
agglomerative_results.round(4)

,Clusters,SilhouetteScore,CalinskiHarabaszScore,DaviesBouldinScore
0,2,0.2674,1562.5776,1.4179
1,3,0.3054,1375.9201,1.1771
2,4,0.3138,1273.8683,1.0620
3,5,0.2465,1248.2888,1.1529
4,6,0.1677,1158.2304,1.2004
5,7,0.1464,1068.3637,1.4943
6,8,0.1368,994.4519,1.4579
7,9,0.1378,943.4885,1.4515
8,10,0.1402,904.9268,1.4094


In [10]:
# Fit the strongest candidate solution from each clustering algorithm.
# These models will now be compared based on cluster size and,
# subsequently, customer behaviour and business interpretability.

kmeans_candidate = KMeans(
    n_clusters=2,
    random_state=42,
    n_init="auto"
)

gmm_candidate = GaussianMixture(
    n_components=2,
    covariance_type="full",
    random_state=42,
    n_init=5,
    max_iter=500
)

agglomerative_candidate = AgglomerativeClustering(
    n_clusters=4,
    linkage="ward"
)

# Generate cluster assignments for each candidate model.
kmeans_labels = kmeans_candidate.fit_predict(X_model)

gmm_candidate.fit(X_model)
gmm_labels = gmm_candidate.predict(X_model)

agglomerative_labels = agglomerative_candidate.fit_predict(
    X_model
)

# Summarise the number and percentage of customers assigned
# to each cluster for every candidate solution.
cluster_size_results = []

candidate_labels = {
    "KMeans_2": kmeans_labels,
    "GMM_2": gmm_labels,
    "Agglomerative_4": agglomerative_labels
}

for model_name, labels in candidate_labels.items():

    cluster_counts = pd.Series(labels).value_counts().sort_index()

    for cluster, count in cluster_counts.items():

        cluster_size_results.append({
            "Model": model_name,
            "Cluster": cluster,
            "CustomerCount": count,
            "CustomerPercent": (
                count / len(X_model) * 100
            )
        })

cluster_size_results = pd.DataFrame(
    cluster_size_results
)

cluster_size_results.round(2)

,Model,Cluster,CustomerCount,CustomerPercent
0,KMeans_2,0,1857,42.85
1,KMeans_2,1,2477,57.15
2,GMM_2,0,338,7.80
3,GMM_2,1,3996,92.20
4,Agglomerative_4,0,2067,47.69
5,Agglomerative_4,1,1892,43.65
6,Agglomerative_4,2,295,6.81
7,Agglomerative_4,3,80,1.85


## 3. Candidate Cluster Profiling

Statistical validation alone does not determine whether a segmentation is
useful.

The strongest candidate solutions are therefore profiled using the original
customer behavioural measures rather than the transformed and scaled modelling
features.

This allows the resulting clusters to be interpreted in meaningful terms such
as recency, purchase frequency, customer value, product breadth, order volume,
postage behaviour and return behaviour.

In [11]:
# Retrieve the full engineered customer dataset so that candidate clusters
# can be interpreted using original business-readable feature values.

customer_features_asset = ml_client.data.get(
    name="online-retail-customer-features",
    version="1"
)

customer_features_table = mltable.load(
    customer_features_asset.path
)

df_customer_profiles = (
    customer_features_table
    .to_pandas_dataframe()
)

print("Customer profile shape:", df_customer_profiles.shape)

Customer profile shape: (4334, 23)


In [12]:
# Create a customer-to-cluster assignment table using the identifiers
# retained alongside the modelling matrix.

cluster_assignments = pd.DataFrame({
    "CustomerID": customer_ids.values,
    "KMeans_2": kmeans_labels,
    "GMM_2": gmm_labels,
    "Agglomerative_4": agglomerative_labels
})

# Join the candidate cluster assignments back to the original
# customer behavioural dataset for business interpretation.

df_candidate_profiles = df_customer_profiles.merge(
    cluster_assignments,
    on="CustomerID",
    how="inner"
)

print("Profiled customers:", len(df_candidate_profiles))

df_candidate_profiles.head()

Profiled customers: 4334


,CustomerID,LastPurchaseDate,Recency,Frequency,MonetaryValue,AverageOrderValue,UniqueProducts,FirstPurchaseDate,CustomerTenureDays,TotalQuantity,...,PostageInvoiceShare,HasPostage,ReturnInvoices,ReturnValue,MerchandiseReturnValue,ObservedMerchandiseReturnValueRate,HasMerchandiseReturn,KMeans_2,GMM_2,Agglomerative_4
0,12346,2011-01-18 10:01:00,326,1,77183.60,77183.600000,1,2011-01-18 10:01:00,0,74215,...,0.0,0,1,77183.6,77183.6,1.0,1,0,1,3
1,12347,2011-12-07 15:52:00,2,7,4310.00,615.714286,103,2010-12-07 14:57:00,365,2458,...,0.0,0,0,0.0,0.0,0.0,0,0,1,0
2,12348,2011-09-25 13:13:00,75,4,1437.24,359.310000,21,2010-12-16 19:09:00,282,2332,...,1.0,1,0,0.0,0.0,0.0,0,0,0,2
3,12349,2011-11-21 09:51:00,19,1,1457.55,1457.550000,72,2011-11-21 09:51:00,0,630,...,1.0,1,0,0.0,0.0,0.0,0,1,0,2
4,12350,2011-02-02 16:01:00,310,1,294.40,294.400000,16,2011-02-02 16:01:00,0,196,...,1.0,1,0,0.0,0.0,0.0,0,1,0,2


In [13]:
# Define the original business-readable features used to interpret
# the candidate customer segments.
profile_features = [
    "Recency",
    "Frequency",
    "MonetaryValue",
    "UniqueProducts",
    "CustomerTenureDays",
    "AverageQuantityPerOrder",
    "PostageInvoiceShare",
    "ObservedMerchandiseReturnValueRate"
]


# Create a reusable function for profiling each clustering solution.
# Median values are used because several customer behaviour measures
# remain highly skewed and can be influenced by extreme customers.
def create_cluster_profile(data, cluster_column):

    profile = (
        data
        .groupby(cluster_column)[profile_features]
        .median()
        .round(2)
    )

    # Add customer counts and population share for each cluster.
    profile.insert(
        0,
        "CustomerCount",
        data.groupby(cluster_column).size()
    )

    profile.insert(
        1,
        "CustomerPercent",
        (
            data.groupby(cluster_column).size()
            / len(data)
            * 100
        ).round(2)
    )

    return profile


# Profile the two-cluster K-Means solution.
kmeans_profile = create_cluster_profile(
    df_candidate_profiles,
    "KMeans_2"
)

print("K-Means - 2 Clusters")
display(kmeans_profile)


# Profile the two-component Gaussian Mixture solution.
gmm_profile = create_cluster_profile(
    df_candidate_profiles,
    "GMM_2"
)

print("Gaussian Mixture Model - 2 Clusters")
display(gmm_profile)


# Profile the four-cluster Agglomerative solution.
agglomerative_profile = create_cluster_profile(
    df_candidate_profiles,
    "Agglomerative_4"
)

print("Agglomerative Clustering - 4 Clusters")
display(agglomerative_profile)

K-Means - 2 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
KMeans_2,,,,,,,,,,
0,1857,42.85,20.0,5.0,1843.83,83.0,267.0,205.33,0.0,0.0
1,2477,57.15,106.0,1.0,332.41,19.0,0.0,129.00,0.0,0.0


Gaussian Mixture Model - 2 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
GMM_2,,,,,,,,,,
0,338,7.8,36.5,3.0,980.40,41.0,123.0,199.50,1.0,0.0
1,3996,92.2,52.0,2.0,643.28,35.0,89.0,157.69,0.0,0.0


Agglomerative Clustering - 4 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
Agglomerative_4,,,,,,,,,,
0,2067,47.69,23.0,4.0,1453.16,72.0,245.0,189.67,0.0,0.0
1,1892,43.65,134.0,1.0,302.23,17.0,0.0,126.00,0.0,0.0
2,295,6.81,42.0,2.0,882.16,37.0,102.0,213.00,1.0,0.0
3,80,1.85,133.5,2.0,524.54,14.0,32.0,114.81,0.0,0.4


## 4. K-Means Segment Structure Review

Although the two-cluster K-Means solution achieved the strongest K-Means
Silhouette Score, it produces a broad high-engagement versus low-engagement
split that may be too coarse for practical customer segmentation.

Additional K-Means solutions from three to seven clusters are therefore
reviewed to determine whether a more granular and business-interpretable
segmentation can be achieved while maintaining acceptable statistical quality.

The candidate solutions are profiled using the original business-readable
customer features so that statistical performance can be considered alongside
business usefulness.

In [14]:
# Profile K-Means solutions from three to seven clusters to assess whether
# a more granular segmentation provides clearer customer behaviours.

kmeans_profile_results = {}

for k in [3, 4, 5, 6, 7]:

    # Train a reproducible K-Means model for the selected cluster count.
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    # Assign each customer to a cluster.
    labels = model.fit_predict(X_model)

    # Add the cluster assignment to a copy of the original
    # business-readable customer dataset.
    profile_data = df_customer_profiles.copy()
    profile_data[f"KMeans_{k}"] = labels

    # Create a median behavioural profile for each cluster.
    profile = create_cluster_profile(
        profile_data,
        f"KMeans_{k}"
    )

    kmeans_profile_results[k] = profile

    print(f"K-Means - {k} Clusters")
    display(profile)

K-Means - 3 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
KMeans_3,,,,,,,,,,
0,1690,38.99,55.0,2.0,734.54,41.0,89.5,222.25,0.0,0.00
1,1451,33.48,165.0,1.0,217.21,12.0,0.0,88.00,0.0,0.00
2,1193,27.53,15.0,7.0,2529.93,103.0,308.0,200.80,0.0,0.01


K-Means - 4 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
KMeans_4,,,,,,,,,,
0,1621,37.40,60.0,2.0,653.38,38.0,61.0,210.0,0.0,0.00
1,1222,28.20,170.0,1.0,202.19,11.0,0.0,79.0,0.0,0.00
2,263,6.07,40.0,2.0,832.39,35.0,102.0,213.0,1.0,0.00
3,1228,28.33,16.0,7.0,2366.90,99.5,299.0,200.4,0.0,0.01


K-Means - 5 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
KMeans_5,,,,,,,,,,
0,1155,26.65,100.0,1.0,428.89,27.0,0.0,222.00,0.0,0.00
1,892,20.58,169.0,1.0,169.80,9.0,0.0,64.25,0.0,0.00
2,672,15.51,10.0,10.0,3763.16,131.5,337.0,227.97,0.0,0.01
3,1355,31.26,32.0,4.0,1031.41,55.0,209.0,160.80,0.0,0.00
4,260,6.00,43.5,2.0,819.70,35.0,97.0,211.00,1.0,0.00


K-Means - 6 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
KMeans_6,,,,,,,,,,
0,1119,25.82,97.0,1.0,426.63,27.0,0.0,221.50,0.0,0.00
1,880,20.30,166.0,1.0,170.10,9.0,0.0,64.00,0.0,0.00
2,669,15.44,9.0,10.0,3740.07,132.0,337.0,227.71,0.0,0.01
3,1350,31.15,32.0,4.0,1024.77,55.0,206.0,162.45,0.0,0.00
4,258,5.95,42.5,2.0,826.17,35.0,100.0,213.64,1.0,0.00
5,58,1.34,125.0,2.0,730.10,14.0,32.0,134.90,0.0,0.48


K-Means - 7 Clusters


,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
KMeans_7,,,,,,,,,,
0,1119,25.82,96.0,1.0,428.89,27.0,0.0,221.00,0.0,0.00
1,869,20.05,166.0,1.0,168.30,9.0,0.0,65.00,0.0,0.00
2,655,15.11,9.0,10.0,3786.24,133.0,338.0,227.71,0.0,0.01
3,1324,30.55,32.0,4.0,1034.79,56.0,208.0,163.00,0.0,0.00
4,256,5.91,43.5,2.0,826.17,35.0,101.5,211.00,1.0,0.00
5,22,0.51,128.0,1.0,307.27,5.0,0.0,87.67,0.0,1.00
6,89,2.05,73.0,3.0,837.10,26.0,137.0,140.00,0.0,0.28


## 5. K-Means Cluster Stability Assessment

The six- and seven-cluster solutions provide the strongest balance between
statistical performance and business interpretability.

Before selecting the final solution, cluster stability is assessed across
multiple random initialisations.

A useful production segmentation should produce broadly consistent customer
groupings rather than changing materially because of different centroid
starting positions.

Adjusted Rand Index (ARI) is used to compare cluster assignments across runs.
An ARI close to 1 indicates highly consistent clustering.

In [15]:
from sklearn.metrics import adjusted_rand_score

# Compare the stability of the leading six- and seven-cluster solutions
# across several random initialisation seeds.

stability_results = []

for k in [6, 7]:

    reference_model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init="auto"
    )

    reference_labels = reference_model.fit_predict(X_model)

    for seed in range(10):

        comparison_model = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init="auto"
        )

        comparison_labels = comparison_model.fit_predict(
            X_model
        )

        stability_results.append({
            "Clusters": k,
            "RandomState": seed,
            "AdjustedRandIndex": adjusted_rand_score(
                reference_labels,
                comparison_labels
            )
        })

# Summarise how consistently each candidate solution reproduces
# the same customer groupings across different initialisations.

stability_results = pd.DataFrame(
    stability_results
)

stability_summary = (
    stability_results
    .groupby("Clusters")["AdjustedRandIndex"]
    .agg(
        MeanARI="mean",
        MinARI="min",
        MaxARI="max",
        StdARI="std"
    )
    .reset_index()
)

stability_summary.round(4)

,Clusters,MeanARI,MinARI,MaxARI,StdARI
0,6,0.8442,0.5126,0.9972,0.1917
1,7,0.8051,0.6505,1.0000,0.1456


### Robust K-Means Stability Check

The initial stability assessment shows generally strong agreement across
random seeds, but some variability remains, particularly for the six-cluster
solution.

Because K-Means can be sensitive to centroid initialisation, the leading
six- and seven-cluster candidates are reassessed using 50 initialisations per
model.

This provides a more robust test of whether the observed customer segments are
reproducible rather than artefacts of a particular starting solution.

In [16]:
# Reassess the stability of the six- and seven-cluster solutions using
# multiple centroid initialisations for each model.

robust_stability_results = []

for k in [6, 7]:

    # Create a robust reference solution using many centroid initialisations.
    reference_model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )

    reference_labels = reference_model.fit_predict(
        X_model
    )

    # Refit the same clustering structure using different random seeds
    # and compare each solution with the reference clustering.
    for seed in range(10):

        comparison_model = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=50
        )

        comparison_labels = comparison_model.fit_predict(
            X_model
        )

        robust_stability_results.append({
            "Clusters": k,
            "RandomState": seed,
            "AdjustedRandIndex": adjusted_rand_score(
                reference_labels,
                comparison_labels
            )
        })


# Summarise the reproducibility of each candidate cluster solution.
robust_stability_results = pd.DataFrame(
    robust_stability_results
)

robust_stability_summary = (
    robust_stability_results
    .groupby("Clusters")["AdjustedRandIndex"]
    .agg(
        MeanARI="mean",
        MinARI="min",
        MaxARI="max",
        StdARI="std"
    )
    .reset_index()
)

robust_stability_summary.round(4)

,Clusters,MeanARI,MinARI,MaxARI,StdARI
0,6,0.9961,0.9933,1.0,0.0017
1,7,0.9957,0.9805,1.0,0.0067


## 6. Final Model Selection

K-Means with six clusters is selected as the final customer segmentation model.

The decision considers statistical performance, cluster stability, business
interpretability and operational suitability.

Although the seven-cluster solution achieved marginally stronger internal
validation scores, it introduced additional fragmentation within a small group
of customers exhibiting unusual return behaviour.

When K-Means was reassessed using 50 centroid initialisations, both candidate
solutions demonstrated very high stability. The six-cluster solution achieved
a mean Adjusted Rand Index of 0.9961 and a minimum of 0.9933, indicating highly
reproducible customer assignments across random initialisations.

The six-cluster model therefore provides the preferred balance between:

- statistical cluster quality;
- reproducibility;
- meaningful behavioural differentiation;
- segment size and interpretability; and
- the ability to assign future customers to existing segments.


In [17]:
# Train the final production-candidate K-Means model using six customer
# segments and multiple centroid initialisations for robust convergence.

final_kmeans = KMeans(
    n_clusters=6,
    random_state=42,
    n_init=50
)

# Fit the model and generate the final cluster assignment for each customer.
final_cluster_labels = final_kmeans.fit_predict(
    X_model
)

# Calculate the final model's internal clustering quality metrics.
final_metrics = evaluate_clustering(
    X_model,
    final_cluster_labels
)

print("Final K-Means Model")
print("-------------------")
print("Clusters:", final_kmeans.n_clusters)
print("Inertia:", round(final_kmeans.inertia_, 4))
print(
    "Silhouette Score:",
    round(final_metrics["SilhouetteScore"], 4)
)
print(
    "Calinski-Harabasz Score:",
    round(final_metrics["CalinskiHarabaszScore"], 4)
)
print(
    "Davies-Bouldin Score:",
    round(final_metrics["DaviesBouldinScore"], 4)
)

Final K-Means Model
-------------------
Clusters: 6
Inertia: 13322.929
Silhouette Score: 0.2218
Calinski-Harabasz Score: 1387.0665
Davies-Bouldin Score: 1.1945


In [18]:
# Associate each customer identifier with the cluster assigned by
# the final six-cluster K-Means model.

final_cluster_assignments = pd.DataFrame({
    "CustomerID": customer_ids.values,
    "Cluster": final_cluster_labels
})

print(
    "Customers assigned:",
    len(final_cluster_assignments)
)

final_cluster_assignments.head()

Customers assigned: 4334


,CustomerID,Cluster
0,12346,4
1,12347,3
2,12348,2
3,12349,2
4,12350,2


In [19]:
# Confirm that every customer received exactly one cluster assignment.

assert len(final_cluster_assignments) == len(customer_ids)
assert final_cluster_assignments["Cluster"].isna().sum() == 0

print("Final cluster assignment validation passed.")

Final cluster assignment validation passed.


## 7. Final Customer Segment Profiling

The final six-cluster K-Means assignments are joined back to the original
business-readable customer features.

Cluster profiling is performed using median values because several behavioural
measures remain skewed and may contain extreme customer values.

The purpose of this analysis is to understand the defining behaviour of each
cluster before assigning meaningful business segment names.

In [20]:
# Join the final cluster assignments back to the original engineered
# customer dataset so that clusters can be interpreted using
# business-readable, unscaled feature values.

df_final_segments = df_customer_profiles.merge(
    final_cluster_assignments,
    on="CustomerID",
    how="inner"
)

print("Customers in final segmentation:", len(df_final_segments))

Customers in final segmentation: 4334


In [21]:
# Define the original behavioural features used to interpret
# the final customer segments.

final_profile_features = [
    "Recency",
    "Frequency",
    "MonetaryValue",
    "UniqueProducts",
    "CustomerTenureDays",
    "AverageQuantityPerOrder",
    "PostageInvoiceShare",
    "ObservedMerchandiseReturnValueRate"
]


# Calculate median behavioural values for each final cluster.

final_cluster_profile = (
    df_final_segments
    .groupby("Cluster")[final_profile_features]
    .median()
    .round(2)
)

In [22]:
# Add cluster size and population share to support interpretation
# of both customer behaviour and segment importance.

cluster_counts = (
    df_final_segments
    .groupby("Cluster")
    .size()
)

final_cluster_profile.insert(
    0,
    "CustomerCount",
    cluster_counts
)

final_cluster_profile.insert(
    1,
    "CustomerPercent",
    (
        cluster_counts
        / len(df_final_segments)
        * 100
    ).round(2)
)

final_cluster_profile

,CustomerCount,CustomerPercent,Recency,Frequency,MonetaryValue,UniqueProducts,CustomerTenureDays,AverageQuantityPerOrder,PostageInvoiceShare,ObservedMerchandiseReturnValueRate
Cluster,,,,,,,,,,
0,1362,31.43,32.5,4.0,1030.78,55.0,205.0,164.00,0.0,0.00
1,847,19.54,165.0,1.0,165.00,9.0,0.0,63.00,0.0,0.00
2,258,5.95,42.5,2.0,826.17,35.0,100.0,213.64,1.0,0.00
3,669,15.44,9.0,10.0,3740.07,132.0,337.0,227.71,0.0,0.01
4,58,1.34,125.0,2.0,730.10,14.0,32.0,134.90,0.0,0.48
5,1140,26.30,105.5,1.0,415.77,27.0,0.0,216.00,0.0,0.00


## 8. Final Segment Interpretation and Naming

The final clusters are assigned descriptive segment names based on their median
behaviour across the eight modelling dimensions.

The segment names are intended to summarise the dominant observed behaviour of
each group rather than imply characteristics that are not present in the data.

- **Cluster 0 – Active Regular Customers:** Relatively recent customers with
  repeat purchasing, moderate-to-high spend, broad product purchasing and an
  established customer history.

- **Cluster 1 – Lapsed Low-Value Customers:** Predominantly one-time customers
  with low spend, limited product breadth and a long period since their last
  purchase.

- **Cluster 2 – Postage-Heavy Occasional Customers:** Moderate-value customers
  whose observed purchases frequently include postage charges.

- **Cluster 3 – High-Value Loyal Customers:** Very recent, frequent and
  long-standing customers with the highest spend and broadest product
  purchasing behaviour.

- **Cluster 4 – High-Return Customers:** A small customer group distinguished
  primarily by a high observed merchandise return-value rate.

- **Cluster 5 – Larger One-Off Customers:** Predominantly one-time customers
  with larger order quantities and higher spend than the low-value one-time
  customer segment.


In [23]:
# Map the numerical cluster identifiers to descriptive customer segment names
# derived from the final behavioural profile.

segment_names = {
    0: "Active Regular Customers",
    1: "Lapsed Low-Value Customers",
    2: "Postage-Heavy Occasional Customers",
    3: "High-Value Loyal Customers",
    4: "High-Return Customers",
    5: "Larger One-Off Customers"
}

# Add the descriptive segment name to each customer's final cluster assignment.

df_final_segments["SegmentName"] = (
    df_final_segments["Cluster"]
    .map(segment_names)
)

In [24]:
# Confirm that every cluster has been mapped to a descriptive segment name.

assert df_final_segments["SegmentName"].isna().sum() == 0

print("Segment naming validation passed.")

Segment naming validation passed.


In [25]:
# Summarise the final customer segmentation using descriptive segment names.

final_segment_summary = (
    df_final_segments
    .groupby(["Cluster", "SegmentName"])
    .size()
    .reset_index(name="CustomerCount")
)

final_segment_summary["CustomerPercent"] = (
    final_segment_summary["CustomerCount"]
    / len(df_final_segments)
    * 100
).round(2)

final_segment_summary

,Cluster,SegmentName,CustomerCount,CustomerPercent
0,0,Active Regular Customers,1362,31.43
1,1,Lapsed Low-Value Customers,847,19.54
2,2,Postage-Heavy Occasional Customers,258,5.95
3,3,High-Value Loyal Customers,669,15.44
4,4,High-Return Customers,58,1.34
5,5,Larger One-Off Customers,1140,26.30


## 9. Extended Segment Profiling

The final customer segments are enriched using additional behavioural measures
that were retained outside the clustering feature set.

These variables did not determine the cluster assignments, but they help explain
the resulting segments in greater business detail.

Median values are used for continuous measures, while binary behavioural
indicators are summarised as the percentage of customers exhibiting the
behaviour.

In [26]:
# Calculate additional median behavioural measures for each final segment.
extended_median_profile = (
    df_final_segments
    .groupby(["Cluster", "SegmentName"])
    .agg(
        AverageOrderValue=("AverageOrderValue", "median"),
        TotalQuantity=("TotalQuantity", "median"),
        AverageDaysBetweenPurchases=("AverageDaysBetweenPurchases", "median"),
        PostageSpend=("PostageSpend", "median"),
        PostageInvoices=("PostageInvoices", "median"),
        ReturnInvoices=("ReturnInvoices", "median"),
        MerchandiseReturnValue=("MerchandiseReturnValue", "median")
    )
    .round(2)
    .reset_index()
)

extended_median_profile

,Cluster,SegmentName,AverageOrderValue,TotalQuantity,AverageDaysBetweenPurchases,PostageSpend,PostageInvoices,ReturnInvoices,MerchandiseReturnValue
0,0,Active Regular Customers,288.38,586.0,71.86,0.0,0.0,0.0,0.00
1,1,Lapsed Low-Value Customers,134.10,78.0,55.63,0.0,0.0,0.0,0.00
2,2,Postage-Heavy Occasional Customers,361.08,504.0,55.99,126.0,2.0,0.0,0.00
3,3,High-Value Loyal Customers,374.75,2159.0,32.92,0.0,0.0,2.0,33.12
4,4,High-Return Customers,307.27,280.0,62.78,0.0,0.0,1.0,363.66
5,5,Larger One-Off Customers,334.18,273.0,41.05,0.0,0.0,0.0,0.00


In [27]:
# Calculate the percentage of customers in each segment who exhibit
# repeat purchasing, postage activity and merchandise return behaviour.

segment_behaviour_rates = (
    df_final_segments
    .groupby(["Cluster", "SegmentName"])
    .agg(
        RepeatCustomerRate=("RepeatCustomer", "mean"),
        HasPostageRate=("HasPostage", "mean"),
        HasMerchandiseReturnRate=("HasMerchandiseReturn", "mean")
    )
    .mul(100)
    .round(2)
    .reset_index()
)

segment_behaviour_rates

,Cluster,SegmentName,RepeatCustomerRate,HasPostageRate,HasMerchandiseReturnRate
0,0,Active Regular Customers,100.00,1.25,40.01
1,1,Lapsed Low-Value Customers,30.34,0.94,10.63
2,2,Postage-Heavy Occasional Customers,65.12,100.00,37.98
3,3,High-Value Loyal Customers,100.00,5.68,75.93
4,4,High-Return Customers,60.34,1.72,100.00
5,5,Larger One-Off Customers,29.65,0.88,18.16


In [30]:
# Combine additional monetary, purchasing, postage and return measures
# into one extended business profile for each customer segment.

extended_segment_profile = (
    extended_median_profile
    .merge(
        segment_behaviour_rates,
        on=["Cluster", "SegmentName"],
        how="left"
    )
)

extended_segment_profile

,Cluster,SegmentName,AverageOrderValue,TotalQuantity,AverageDaysBetweenPurchases,PostageSpend,PostageInvoices,ReturnInvoices,MerchandiseReturnValue,RepeatCustomerRate,HasPostageRate,HasMerchandiseReturnRate
0,0,Active Regular Customers,288.38,586.0,71.86,0.0,0.0,0.0,0.00,100.00,1.25,40.01
1,1,Lapsed Low-Value Customers,134.10,78.0,55.63,0.0,0.0,0.0,0.00,30.34,0.94,10.63
2,2,Postage-Heavy Occasional Customers,361.08,504.0,55.99,126.0,2.0,0.0,0.00,65.12,100.00,37.98
3,3,High-Value Loyal Customers,374.75,2159.0,32.92,0.0,0.0,2.0,33.12,100.00,5.68,75.93
4,4,High-Return Customers,307.27,280.0,62.78,0.0,0.0,1.0,363.66,60.34,1.72,100.00
5,5,Larger One-Off Customers,334.18,273.0,41.05,0.0,0.0,0.0,0.00,29.65,0.88,18.16


In [29]:
# Refine the final segment names based on the extended behavioural profile.

segment_names = {
    0: "Active Regular Customers",
    1: "Lapsed Low-Value Customers",
    2: "Postage-Heavy Occasional Customers",
    3: "High-Value Loyal Customers",
    4: "High-Return Customers",
    5: "Larger-Basket Low-Frequency Customers"
}

# Apply the refined business names to the final customer segmentation.

df_final_segments["SegmentName"] = (
    df_final_segments["Cluster"]
    .map(segment_names)
)

### Extended profile interpretation

The extended behavioural profile supports the overall segment definitions.

Cluster 5 is renamed **Larger-Basket Low-Frequency Customers** rather than
Larger One-Off Customers. Although the median customer has only one purchase,
approximately 30% of customers in the segment are repeat purchasers. The revised
name therefore better reflects the combination of low purchase frequency and
relatively large order sizes.

`AverageDaysBetweenPurchases` should be interpreted only for customers with
multiple observed purchases. Single-purchase customers have no purchase interval,
so the segment-level median for this measure represents the repeat-purchasing
subset rather than the entire segment.

## 10. Persist Final Model and Customer Segment Assignments

The selected six-cluster K-Means model is persisted for reproducible use in
future scoring and deployment workflows.

Final customer-to-segment assignments are also saved separately so that segment
membership can be used for analysis, reporting and downstream business
applications.

Model metadata is retained alongside the trained model to document the modelling
configuration, feature order, evaluation metrics and segment definitions.

In [31]:
import os
import json
import joblib

# Create dedicated locations for the trained model and final segment dataset.
model_artifact_folder = "../deployment/model"
os.makedirs(model_artifact_folder, exist_ok=True)

customer_segments_path = (
    "../data/processed/online_retail_customer_segments.parquet"
)

In [32]:
# Persist the trained six-cluster K-Means model, including the learned
# cluster centroids required to assign future customers to segments.

final_model_path = os.path.join(
    model_artifact_folder,
    "final_kmeans_model.joblib"
)

joblib.dump(
    final_kmeans,
    final_model_path
)

print(f"Final model saved to: {final_model_path}")

Final model saved to: ../deployment/model/final_kmeans_model.joblib


In [33]:
# Persist the final customer-to-segment mapping.
# Only identifiers and final segment information are required here.

customer_segment_assignments = (
    df_final_segments[
        [
            "CustomerID",
            "Cluster",
            "SegmentName"
        ]
    ]
    .copy()
)

customer_segment_assignments.to_parquet(
    customer_segments_path,
    engine="pyarrow",
    index=False
)

print(
    f"Customer segment assignments saved to: "
    f"{customer_segments_path}"
)

Customer segment assignments saved to: ../data/processed/online_retail_customer_segments.parquet


In [34]:
# Record the final modelling configuration and evaluation results
# so the trained artefact can be understood independently of the notebook.

model_metadata = {
    "model_type": "KMeans",
    "n_clusters": 6,
    "random_state": 42,
    "n_init": 50,
    "training_customers": len(X_model),
    "model_features": list(X_model.columns),
    "silhouette_score": float(
        final_metrics["SilhouetteScore"]
    ),
    "calinski_harabasz_score": float(
        final_metrics["CalinskiHarabaszScore"]
    ),
    "davies_bouldin_score": float(
        final_metrics["DaviesBouldinScore"]
    ),
    "segment_names": segment_names
}

model_metadata_path = os.path.join(
    model_artifact_folder,
    "model_metadata.json"
)

with open(model_metadata_path, "w") as file:
    json.dump(
        model_metadata,
        file,
        indent=4
    )

print(f"Model metadata saved to: {model_metadata_path}")

Model metadata saved to: ../deployment/model/model_metadata.json


In [35]:
# Confirm that the expected production modelling artefacts exist.

print(
    "Model artefacts:",
    os.listdir(model_artifact_folder)
)

print(
    "Segment assignment shape:",
    pd.read_parquet(customer_segments_path).shape
)

Model artefacts: ['final_kmeans_model.joblib', 'model_metadata.json']
Segment assignment shape: (4334, 3)


## 11. Register Final Model in Azure ML

The selected six-cluster K-Means model is registered as a versioned Azure ML
model asset.

The registered artefact contains both the trained model and supporting metadata,
providing a governed and reproducible model version for downstream scoring and
deployment.

In [36]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# Define the final customer segmentation model as a versioned
# Azure ML custom model asset.
final_model_asset = Model(
    name="online-retail-kmeans-segmentation",
    version="1",
    type=AssetTypes.CUSTOM_MODEL,
    path=model_artifact_folder,
    description=(
        "Six-cluster K-Means customer segmentation model for the "
        "Online Retail customer segmentation project."
    )
)

# Register the trained model and its metadata in the Azure ML workspace.
registered_final_model = ml_client.models.create_or_update(
    final_model_asset
)

In [38]:
print("Name:", registered_final_model.name)
print("Version:", registered_final_model.version)
print("Type:", registered_final_model.type)
print("Path:", registered_final_model.path)

Name: online-retail-kmeans-segmentation
Version: 1
Type: custom_model
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourceGroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/583ea73d78fca78932cbc58aa41f991b25a31fc9e57906442f91910d09ae79a7/model


## 12. Register Customer Segment Assignments

The final customer-to-segment assignments are registered as a versioned Azure ML
data asset.

This provides a governed output containing each customer identifier, numerical
cluster assignment and descriptive segment name for downstream reporting,
analysis and business use.

In [39]:
# Register the final customer segment assignment dataset as a versioned
# Azure ML URI_FILE data asset.

customer_segments_asset = Data(
    name="online-retail-customer-segments-parquet",
    version="1",
    type=AssetTypes.URI_FILE,
    path=customer_segments_path,
    description=(
        "Final customer-to-segment assignments produced by the "
        "six-cluster Online Retail K-Means segmentation model."
    )
)

# Upload and register the persisted customer segment assignments
# in the Azure ML workspace.

registered_customer_segments = ml_client.data.create_or_update(
    customer_segments_asset
)

In [41]:
print("Name:", registered_customer_segments.name)
print("Version:", registered_customer_segments.version)
print("Type:", registered_customer_segments.type)
print("Path:", registered_customer_segments.path)

Name: online-retail-customer-segments-parquet
Version: 1
Type: uri_file
Path: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/2bad1e874436e27c050d891407770f0cacda92cdc26c070b8a1b955be00d9ff9/online_retail_customer_segments.parquet


## 13. Create Customer Segment MLTable

The registered customer segment Parquet asset is exposed through an MLTable so
that downstream reporting, analysis and future Azure ML workflows can consume
the final segmentation output consistently.

In [42]:
# Create an MLTable definition that points to the registered
# customer segment assignment Parquet dataset in Azure storage.

customer_segments_table = mltable.from_parquet_files(
    paths=[
        {
            "file": registered_customer_segments.path
        }
    ]
)

# Materialise the MLTable to confirm that the final customer
# segment assignments can be read successfully.

customer_segments_test = (
    customer_segments_table
    .to_pandas_dataframe()
)

print(
    "Customer segments MLTable shape:",
    customer_segments_test.shape
)

Customer segments MLTable shape: (4334, 3)


In [43]:
# Create a project folder for the final customer-segment MLTable definition.

customer_segments_mltable_folder = "../data/mltable/customer-segments"

os.makedirs(
    customer_segments_mltable_folder,
    exist_ok=True
)

# Save the MLTable definition file for the final customer segment dataset.

customer_segments_table.save(
    customer_segments_mltable_folder
)

paths:
- file: azureml://subscriptions/9d0e4acf-2675-4581-90be-c8f45f73d333/resourcegroups/clustering-project/workspaces/clustering-workspace/datastores/workspaceblobstore/paths/LocalUpload/2bad1e874436e27c050d891407770f0cacda92cdc26c070b8a1b955be00d9ff9/online_retail_customer_segments.parquet
transformations:
- read_parquet:
    include_path_column: false
    path_column: Path
type: mltable

## 14. Register Final Customer Segment MLTable

The final customer segmentation output is registered as an MLTable so it can be
consumed consistently by downstream Azure ML workflows, reporting processes and
future analytical applications.

This asset contains the customer identifier, numerical cluster assignment and
descriptive segment name produced by the selected six-cluster K-Means model.

In [44]:
# Define the final customer segment MLTable as a versioned Azure ML data asset.

customer_segments_mltable_asset = Data(
    name="online-retail-customer-segments",
    version="1",
    type=AssetTypes.MLTABLE,
    path=customer_segments_mltable_folder,
    description=(
        "Final customer segment assignments produced by the "
        "six-cluster Online Retail K-Means segmentation model."
    )
)

# Register the final customer segmentation MLTable in the Azure ML workspace.

registered_customer_segments_mltable = ml_client.data.create_or_update(
    customer_segments_mltable_asset
)

Uploading customer-segments (0.0 MBs): 100%|██████████| 395/395 [00:00<00:00, 19553.52it/s]




In [45]:
print("Name:", registered_customer_segments_mltable.name)
print("Version:", registered_customer_segments_mltable.version)
print("Type:", registered_customer_segments_mltable.type)

Name: online-retail-customer-segments
Version: 1
Type: mltable
